# ACDADA — Notebook 08: Self-Evaluation Agent

**Autonomous Performance Monitoring & Drift Detection**

This notebook implements:
1. Performance evaluation framework for all ACDADA agents
2. Concept drift detection (ADWIN, Page-Hinkley, DDM)
3. Deception effectiveness metrics
4. Automated health scoring and alerting
5. Retraining trigger logic

In [ ]:
# ============================================================
# DATASET / REFERENCE LINKS
# ============================================================
#
# This notebook evaluates models trained in Notebooks 02-07.
# It uses the processed data from Notebook 01 and model outputs
# from all previous notebooks.
#
# Drift detection references:
#   ADWIN: Bifet & Gavalda (2007)
#   DDM: Gama et al. (2004)
#   Page-Hinkley: Page (1954)
# ============================================================

## 1. Imports & Configuration

In [ ]:
import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from typing import Dict, List, Tuple, Optional, Any
from dataclasses import dataclass, field
from collections import deque
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, confusion_matrix, matthews_corrcoef,
    classification_report
)
from scipy.stats import ks_2samp, chi2_contingency
import json
import time
import warnings

warnings.filterwarnings('ignore')
np.random.seed(42)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
MODELS_DIR = Path('../models')
EVAL_DIR = Path('../models/evaluation')
EVAL_DIR.mkdir(parents=True, exist_ok=True)

print(f'Device: {DEVICE}')

---
## 2. Agent Performance Metrics

In [ ]:
@dataclass
class AgentMetrics:
    """Metrics container for a single agent evaluation."""
    agent_name: str
    timestamp: float = field(default_factory=time.time)
    accuracy: float = 0.0
    precision: float = 0.0
    recall: float = 0.0
    f1: float = 0.0
    auc_roc: float = 0.0
    mcc: float = 0.0
    fpr: float = 0.0  # False positive rate
    latency_ms: float = 0.0  # Inference latency
    throughput: float = 0.0  # Samples/sec
    custom_metrics: Dict[str, float] = field(default_factory=dict)
    
    def health_score(self) -> float:
        """Compute overall health score (0-100)."""
        weights = {
            'f1': 0.25,
            'recall': 0.25,  # Especially important for security
            'auc_roc': 0.20,
            'mcc': 0.15,
            'fpr_inv': 0.15,  # Lower FPR = better
        }
        
        score = (
            weights['f1'] * self.f1 +
            weights['recall'] * self.recall +
            weights['auc_roc'] * self.auc_roc +
            weights['mcc'] * (self.mcc + 1) / 2 +  # MCC is [-1,1]
            weights['fpr_inv'] * (1 - self.fpr)
        ) * 100
        
        return np.clip(score, 0, 100)
    
    def to_dict(self) -> Dict:
        return {
            'agent_name': self.agent_name,
            'timestamp': self.timestamp,
            'accuracy': self.accuracy,
            'precision': self.precision,
            'recall': self.recall,
            'f1': self.f1,
            'auc_roc': self.auc_roc,
            'mcc': self.mcc,
            'fpr': self.fpr,
            'latency_ms': self.latency_ms,
            'throughput': self.throughput,
            'health_score': self.health_score(),
            **self.custom_metrics,
        }


class AgentEvaluator:
    """
    Evaluates any ACDADA agent.
    Supports binary detection, multi-class classification,
    anomaly detection, and RL agent evaluation.
    """
    
    def evaluate_classifier(self, agent_name: str, y_true: np.ndarray,
                            y_pred: np.ndarray, y_prob: Optional[np.ndarray] = None,
                            latency_ms: float = 0.0) -> AgentMetrics:
        """Evaluate a classification agent."""
        is_binary = len(np.unique(y_true)) <= 2
        avg = 'binary' if is_binary else 'weighted'
        
        metrics = AgentMetrics(agent_name=agent_name)
        metrics.accuracy = accuracy_score(y_true, y_pred)
        metrics.precision = precision_score(y_true, y_pred, average=avg, zero_division=0)
        metrics.recall = recall_score(y_true, y_pred, average=avg, zero_division=0)
        metrics.f1 = f1_score(y_true, y_pred, average=avg, zero_division=0)
        metrics.mcc = matthews_corrcoef(y_true, y_pred)
        metrics.latency_ms = latency_ms
        
        if y_prob is not None:
            try:
                if is_binary:
                    metrics.auc_roc = roc_auc_score(y_true, y_prob)
                else:
                    metrics.auc_roc = roc_auc_score(y_true, y_prob, multi_class='ovr', average='weighted')
            except ValueError:
                metrics.auc_roc = 0.0
        
        # False positive rate
        if is_binary:
            cm = confusion_matrix(y_true, y_pred)
            if cm.shape == (2, 2):
                tn, fp, fn, tp = cm.ravel()
                metrics.fpr = fp / (fp + tn) if (fp + tn) > 0 else 0.0
        
        return metrics
    
    def evaluate_anomaly_detector(self, agent_name: str,
                                   y_true: np.ndarray, scores: np.ndarray,
                                   threshold: float) -> AgentMetrics:
        """Evaluate anomaly detection agent."""
        y_pred = (scores > threshold).astype(int)
        metrics = self.evaluate_classifier(agent_name, y_true, y_pred, scores)
        
        # Additional anomaly metrics
        metrics.custom_metrics['threshold'] = threshold
        metrics.custom_metrics['mean_score_normal'] = float(scores[y_true == 0].mean()) if (y_true == 0).any() else 0
        metrics.custom_metrics['mean_score_anomaly'] = float(scores[y_true == 1].mean()) if (y_true == 1).any() else 0
        metrics.custom_metrics['score_separation'] = (
            metrics.custom_metrics['mean_score_anomaly'] - metrics.custom_metrics['mean_score_normal']
        )
        
        return metrics
    
    def evaluate_rl_agent(self, agent_name: str, episode_rewards: np.ndarray,
                          hp_interactions: np.ndarray, real_compromises: np.ndarray,
                          detection_times: np.ndarray) -> AgentMetrics:
        """Evaluate RL deception agent."""
        metrics = AgentMetrics(agent_name=agent_name)
        
        # RL-specific metrics
        metrics.custom_metrics['mean_reward'] = float(episode_rewards.mean())
        metrics.custom_metrics['std_reward'] = float(episode_rewards.std())
        metrics.custom_metrics['mean_hp_interactions'] = float(hp_interactions.mean())
        metrics.custom_metrics['mean_real_compromises'] = float(real_compromises.mean())
        metrics.custom_metrics['deception_rate'] = float(
            hp_interactions.sum() / max(hp_interactions.sum() + real_compromises.sum(), 1)
        )
        
        valid_dt = detection_times[detection_times > 0]
        metrics.custom_metrics['mean_detection_time'] = float(valid_dt.mean()) if len(valid_dt) > 0 else -1
        metrics.custom_metrics['detection_rate'] = float(len(valid_dt) / len(detection_times))
        
        # Compute a pseudo-F1 for RL:
        dr = metrics.custom_metrics['deception_rate']
        det_rate = metrics.custom_metrics['detection_rate']
        metrics.f1 = 2 * dr * det_rate / (dr + det_rate) if (dr + det_rate) > 0 else 0
        metrics.recall = det_rate
        metrics.precision = dr
        
        return metrics

evaluator = AgentEvaluator()
print('AgentEvaluator defined.')

---
## 3. Concept Drift Detection

In [ ]:
class ADWINDriftDetector:
    """
    ADWIN (Adaptive Windowing) drift detector.
    Maintains a variable-length window and detects if the mean
    of two sub-windows differs significantly.
    """
    
    def __init__(self, delta: float = 0.002):
        self.delta = delta
        self.window = deque()
        self.total = 0.0
        self.variance = 0.0
        self.width = 0
        self.drift_count = 0
    
    def update(self, value: float) -> bool:
        """Add a value and check for drift. Returns True if drift detected."""
        self.window.append(value)
        self.total += value
        self.width += 1
        
        if self.width < 10:
            return False
        
        drift = self._check_drift()
        if drift:
            self.drift_count += 1
        return drift
    
    def _check_drift(self) -> bool:
        """Check if there's a significant difference in sub-windows."""
        window_list = list(self.window)
        n = len(window_list)
        
        for split in range(max(5, n // 4), n - max(5, n // 4)):
            w0 = np.array(window_list[:split])
            w1 = np.array(window_list[split:])
            
            n0, n1 = len(w0), len(w1)
            mu0, mu1 = w0.mean(), w1.mean()
            
            # Hoeffding bound
            m = 1.0 / (1.0 / n0 + 1.0 / n1)
            epsilon = np.sqrt(np.log(4.0 / self.delta) / (2.0 * m))
            
            if abs(mu0 - mu1) >= epsilon:
                # Drift detected — shrink window
                for _ in range(split):
                    removed = self.window.popleft()
                    self.total -= removed
                    self.width -= 1
                return True
        
        return False


class PageHinkleyDetector:
    """
    Page-Hinkley test for change detection.
    Detects both upward and downward shifts in mean.
    """
    
    def __init__(self, delta: float = 0.005, threshold: float = 50.0,
                 alpha: float = 0.9999):
        self.delta = delta
        self.threshold = threshold
        self.alpha = alpha
        self.sum = 0.0
        self.x_mean = 0.0
        self.n = 0
        self.m_t = 0.0
        self.M_t = 0.0
        self.drift_count = 0
    
    def update(self, value: float) -> bool:
        self.n += 1
        self.x_mean = self.x_mean + (value - self.x_mean) / self.n
        self.sum = self.alpha * self.sum + (value - self.x_mean - self.delta)
        self.m_t = min(self.m_t, self.sum)
        self.M_t = max(self.M_t, self.sum)
        
        drift = (self.M_t - self.m_t) > self.threshold
        if drift:
            self.drift_count += 1
            # Reset
            self.sum = 0.0
            self.m_t = 0.0
            self.M_t = 0.0
        return drift


class DDMDetector:
    """
    Drift Detection Method (DDM) by Gama et al.
    Monitors error rate and its standard deviation.
    """
    
    def __init__(self, warning_level: float = 2.0, drift_level: float = 3.0,
                 min_samples: int = 30):
        self.warning_level = warning_level
        self.drift_level = drift_level
        self.min_samples = min_samples
        self.n = 0
        self.p = 0.0
        self.s = 0.0
        self.p_min = float('inf')
        self.s_min = float('inf')
        self.state = 'stable'  # 'stable', 'warning', 'drift'
        self.drift_count = 0
    
    def update(self, is_error: bool) -> str:
        """Update with prediction result. Returns state."""
        self.n += 1
        self.p = self.p + (float(is_error) - self.p) / self.n
        self.s = np.sqrt(self.p * (1 - self.p) / self.n)
        
        if self.n < self.min_samples:
            return 'stable'
        
        if self.p + self.s < self.p_min + self.s_min:
            self.p_min = self.p
            self.s_min = self.s
        
        if self.p + self.s > self.p_min + self.drift_level * self.s_min:
            self.state = 'drift'
            self.drift_count += 1
            # Reset
            self.n = 0; self.p = 0; self.s = 0
            self.p_min = float('inf'); self.s_min = float('inf')
        elif self.p + self.s > self.p_min + self.warning_level * self.s_min:
            self.state = 'warning'
        else:
            self.state = 'stable'
        
        return self.state


class FeatureDriftDetector:
    """
    Detect distribution drift in input features using
    Kolmogorov-Smirnov test.
    """
    
    def __init__(self, reference_data: np.ndarray, p_threshold: float = 0.01):
        self.reference = reference_data
        self.p_threshold = p_threshold
        self.n_features = reference_data.shape[1] if reference_data.ndim > 1 else 1
    
    def check_drift(self, new_data: np.ndarray) -> Dict:
        """Check if new data has drifted from reference."""
        if new_data.ndim == 1:
            new_data = new_data.reshape(-1, 1)
            ref = self.reference.reshape(-1, 1)
        else:
            ref = self.reference
        
        n_drifted = 0
        drift_details = []
        
        for i in range(min(ref.shape[1], new_data.shape[1])):
            stat, p_value = ks_2samp(ref[:, i], new_data[:, i])
            is_drift = p_value < self.p_threshold
            if is_drift:
                n_drifted += 1
            drift_details.append({
                'feature_idx': i,
                'ks_stat': float(stat),
                'p_value': float(p_value),
                'drift': is_drift,
            })
        
        total_features = min(ref.shape[1], new_data.shape[1])
        drift_ratio = n_drifted / total_features if total_features > 0 else 0
        
        return {
            'n_features_drifted': n_drifted,
            'total_features': total_features,
            'drift_ratio': drift_ratio,
            'overall_drift': drift_ratio > 0.1,  # >10% features drifted
            'details': drift_details,
        }

print('Drift detectors defined: ADWIN, PageHinkley, DDM, FeatureDrift')

---
## 4. Simulated Evaluation (with synthetic data)

In [ ]:
def simulate_agent_predictions(n_samples=2000, drift_point=1200, base_accuracy=0.92):
    """
    Simulate agent predictions with concept drift.
    Before drift_point: high accuracy
    After drift_point: degraded accuracy
    """
    y_true = np.random.randint(0, 2, size=n_samples)
    y_pred = y_true.copy()
    y_prob = np.zeros(n_samples)
    
    for i in range(n_samples):
        if i < drift_point:
            # Normal performance
            if np.random.random() > base_accuracy:
                y_pred[i] = 1 - y_pred[i]  # flip
            y_prob[i] = 0.9 if y_true[i] == 1 else 0.1
            y_prob[i] += np.random.normal(0, 0.05)
        else:
            # Degraded performance (drift)
            degraded_acc = base_accuracy - 0.2 * ((i - drift_point) / (n_samples - drift_point))
            if np.random.random() > degraded_acc:
                y_pred[i] = 1 - y_pred[i]
            y_prob[i] = 0.7 if y_true[i] == 1 else 0.3
            y_prob[i] += np.random.normal(0, 0.15)
    
    y_prob = np.clip(y_prob, 0, 1)
    return y_true, y_pred, y_prob


# Generate simulated data for 3 agents
agents_data = {
    'ThreatDetector': simulate_agent_predictions(2000, 1200, 0.94),
    'AnomalyDetector': simulate_agent_predictions(2000, 1000, 0.88),
    'AttackClassifier': simulate_agent_predictions(2000, 1500, 0.91),
}

print('Simulated agent prediction data generated.')

In [ ]:
# Evaluate each agent
all_metrics = {}

for agent_name, (y_true, y_pred, y_prob) in agents_data.items():
    # Overall evaluation
    metrics = evaluator.evaluate_classifier(agent_name, y_true, y_pred, y_prob)
    all_metrics[agent_name] = metrics
    
    print(f'\n{"="*50}')
    print(f'Agent: {agent_name}')
    print(f'{"="*50}')
    print(f'  Accuracy:  {metrics.accuracy:.4f}')
    print(f'  Precision: {metrics.precision:.4f}')
    print(f'  Recall:    {metrics.recall:.4f}')
    print(f'  F1:        {metrics.f1:.4f}')
    print(f'  AUC-ROC:   {metrics.auc_roc:.4f}')
    print(f'  MCC:       {metrics.mcc:.4f}')
    print(f'  FPR:       {metrics.fpr:.4f}')
    print(f'  Health:    {metrics.health_score():.1f}/100')

---
## 5. Drift Detection on Agent Predictions

In [ ]:
def run_drift_detection(y_true, y_pred, agent_name, window_size=50):
    """
    Run all drift detectors on agent prediction stream.
    Returns drift points for each detector.
    """
    adwin = ADWINDriftDetector(delta=0.002)
    ph = PageHinkleyDetector(delta=0.005, threshold=30.0)
    ddm = DDMDetector(warning_level=2.0, drift_level=3.0)
    
    errors = (y_true != y_pred).astype(float)
    rolling_accuracy = []
    adwin_drifts = []
    ph_drifts = []
    ddm_drifts = []
    ddm_warnings = []
    
    for i, error in enumerate(errors):
        # Rolling accuracy
        start = max(0, i - window_size)
        rolling_accuracy.append(1.0 - errors[start:i+1].mean())
        
        # ADWIN
        if adwin.update(error):
            adwin_drifts.append(i)
        
        # Page-Hinkley
        if ph.update(error):
            ph_drifts.append(i)
        
        # DDM
        state = ddm.update(bool(error))
        if state == 'drift':
            ddm_drifts.append(i)
        elif state == 'warning':
            ddm_warnings.append(i)
    
    return {
        'rolling_accuracy': np.array(rolling_accuracy),
        'adwin_drifts': adwin_drifts,
        'ph_drifts': ph_drifts,
        'ddm_drifts': ddm_drifts,
        'ddm_warnings': ddm_warnings,
    }


# Run drift detection for all agents
drift_results = {}
for agent_name, (y_true, y_pred, _) in agents_data.items():
    drift_results[agent_name] = run_drift_detection(y_true, y_pred, agent_name)
    n_adwin = len(drift_results[agent_name]['adwin_drifts'])
    n_ph = len(drift_results[agent_name]['ph_drifts'])
    n_ddm = len(drift_results[agent_name]['ddm_drifts'])
    print(f'{agent_name}: ADWIN={n_adwin} drifts, PH={n_ph} drifts, DDM={n_ddm} drifts')

In [ ]:
# Visualize drift detection
fig, axes = plt.subplots(len(drift_results), 1, figsize=(16, 5*len(drift_results)))
if len(drift_results) == 1: axes = [axes]

for ax, (agent_name, result) in zip(axes, drift_results.items()):
    x = np.arange(len(result['rolling_accuracy']))
    
    # Plot rolling accuracy
    ax.plot(x, result['rolling_accuracy'], color='#2196F3', linewidth=1, alpha=0.8, label='Rolling Accuracy')
    
    # Mark drifts
    for d in result['adwin_drifts']:
        ax.axvline(d, color='red', alpha=0.5, linestyle='--', linewidth=1)
    for d in result['ph_drifts']:
        ax.axvline(d, color='orange', alpha=0.5, linestyle=':', linewidth=1)
    for d in result['ddm_drifts']:
        ax.axvline(d, color='purple', alpha=0.5, linestyle='-.', linewidth=1)
    
    # Legend entries
    ax.axvline(-1, color='red', alpha=0.5, linestyle='--', label=f'ADWIN ({len(result["adwin_drifts"])})')
    ax.axvline(-1, color='orange', alpha=0.5, linestyle=':', label=f'PageHinkley ({len(result["ph_drifts"])})')
    ax.axvline(-1, color='purple', alpha=0.5, linestyle='-.', label=f'DDM ({len(result["ddm_drifts"])})')
    
    ax.set_title(f'{agent_name} — Rolling Accuracy & Drift Points', fontsize=12, fontweight='bold')
    ax.set_xlabel('Sample'); ax.set_ylabel('Accuracy')
    ax.legend(loc='lower left'); ax.grid(True, alpha=0.3)
    ax.set_ylim(0.5, 1.05)

plt.suptitle('Concept Drift Detection Across Agents', fontsize=14, fontweight='bold')
plt.tight_layout(); plt.show()

---
## 6. Deception Effectiveness Metrics

In [ ]:
@dataclass
class DeceptionMetrics:
    """Metrics specific to deception effectiveness."""
    deception_success_rate: float = 0.0  # % attackers interacted with honeypots
    mean_attacker_dwell_time: float = 0.0  # Avg time attacker spent on honeypots
    real_asset_protection_rate: float = 0.0  # % real assets uncompromised
    false_alarm_rate: float = 0.0
    detection_latency: float = 0.0  # Avg time to first detection
    coverage_ratio: float = 0.0  # % of attack surface covered by deception
    attacker_confusion_score: float = 0.0  # How much attacker was misled
    
    def overall_effectiveness(self) -> float:
        """Compute weighted deception effectiveness score."""
        return (
            0.30 * self.deception_success_rate +
            0.25 * self.real_asset_protection_rate +
            0.20 * (1 - self.false_alarm_rate) +
            0.15 * self.attacker_confusion_score +
            0.10 * min(1.0, self.mean_attacker_dwell_time / 20.0)  # Normalized
        ) * 100


def compute_deception_metrics(episode_infos: List[Dict]) -> DeceptionMetrics:
    """
    Compute deception metrics from RL episode results.
    """
    metrics = DeceptionMetrics()
    
    if not episode_infos:
        return metrics
    
    hp_interactions = np.array([e.get('honeypot_interactions', 0) for e in episode_infos])
    real_compromises = np.array([e.get('real_compromises', 0) for e in episode_infos])
    detection_times = np.array([e.get('detection_time', -1) for e in episode_infos])
    atk_times = np.array([e.get('attacker_time_in_network', 0) for e in episode_infos])
    
    total_interactions = hp_interactions.sum() + real_compromises.sum()
    metrics.deception_success_rate = float(hp_interactions.sum() / max(total_interactions, 1))
    metrics.mean_attacker_dwell_time = float(atk_times.mean())
    metrics.real_asset_protection_rate = float((real_compromises == 0).mean())
    
    valid_dt = detection_times[detection_times > 0]
    metrics.detection_latency = float(valid_dt.mean()) if len(valid_dt) > 0 else float('inf')
    
    # Confusion score: ratio of honeypot interactions to total
    confusion_per_ep = hp_interactions / np.maximum(hp_interactions + real_compromises, 1)
    metrics.attacker_confusion_score = float(confusion_per_ep.mean())
    
    return metrics


# Simulate episode results for testing
simulated_episodes = []
for i in range(200):
    simulated_episodes.append({
        'honeypot_interactions': np.random.randint(0, 5),
        'real_compromises': np.random.randint(0, 3),
        'detection_time': np.random.randint(1, 50) if np.random.random() > 0.2 else -1,
        'attacker_time_in_network': np.random.randint(5, 80),
    })

dec_metrics = compute_deception_metrics(simulated_episodes)
print(f'Deception Success Rate:     {dec_metrics.deception_success_rate:.3f}')
print(f'Attacker Dwell Time:        {dec_metrics.mean_attacker_dwell_time:.1f} steps')
print(f'Real Asset Protection:      {dec_metrics.real_asset_protection_rate:.3f}')
print(f'Detection Latency:          {dec_metrics.detection_latency:.1f} steps')
print(f'Attacker Confusion:         {dec_metrics.attacker_confusion_score:.3f}')
print(f'Overall Effectiveness:      {dec_metrics.overall_effectiveness():.1f}/100')

---
## 7. Self-Evaluation Dashboard

In [ ]:
class SelfEvaluationAgent:
    """
    Autonomous self-evaluation agent.
    Monitors all ACDADA agents and triggers alerts/retraining.
    """
    
    # Thresholds
    HEALTH_CRITICAL = 60
    HEALTH_WARNING = 75
    DRIFT_THRESHOLD = 3  # Max drift events before alert
    
    def __init__(self):
        self.evaluator = AgentEvaluator()
        self.history: Dict[str, List[AgentMetrics]] = {}
        self.drift_detectors: Dict[str, Dict] = {}
        self.alerts: List[Dict] = []
    
    def register_agent(self, agent_name: str):
        """Register an agent for monitoring."""
        self.history[agent_name] = []
        self.drift_detectors[agent_name] = {
            'adwin': ADWINDriftDetector(),
            'ph': PageHinkleyDetector(),
            'ddm': DDMDetector(),
        }
    
    def evaluate_and_record(self, agent_name: str, y_true: np.ndarray,
                            y_pred: np.ndarray, y_prob: Optional[np.ndarray] = None) -> Dict:
        """
        Evaluate agent, check for drift, and generate status report.
        """
        if agent_name not in self.history:
            self.register_agent(agent_name)
        
        # Evaluate
        metrics = self.evaluator.evaluate_classifier(agent_name, y_true, y_pred, y_prob)
        self.history[agent_name].append(metrics)
        
        # Check drift
        errors = (y_true != y_pred).astype(float)
        drift_events = 0
        for error in errors:
            for detector in self.drift_detectors[agent_name].values():
                if hasattr(detector, 'update'):
                    result = detector.update(error)
                    if result is True or result == 'drift':
                        drift_events += 1
        
        # Generate status
        health = metrics.health_score()
        status = 'healthy'
        
        if health < self.HEALTH_CRITICAL:
            status = 'critical'
            self._add_alert(agent_name, 'CRITICAL', 
                          f'Health score {health:.1f} below critical threshold {self.HEALTH_CRITICAL}')
        elif health < self.HEALTH_WARNING:
            status = 'warning'
            self._add_alert(agent_name, 'WARNING',
                          f'Health score {health:.1f} below warning threshold {self.HEALTH_WARNING}')
        
        if drift_events > self.DRIFT_THRESHOLD:
            if status != 'critical':
                status = 'warning'
            self._add_alert(agent_name, 'DRIFT',
                          f'{drift_events} drift events detected — consider retraining')
        
        report = {
            'agent_name': agent_name,
            'status': status,
            'health_score': health,
            'metrics': metrics.to_dict(),
            'drift_events': drift_events,
            'needs_retraining': status == 'critical' or drift_events > self.DRIFT_THRESHOLD * 2,
            'n_evaluations': len(self.history[agent_name]),
        }
        
        return report
    
    def _add_alert(self, agent_name: str, level: str, message: str):
        alert = {
            'timestamp': time.time(),
            'agent': agent_name,
            'level': level,
            'message': message,
        }
        self.alerts.append(alert)
    
    def get_system_status(self) -> Dict:
        """Get overall system health status."""
        agent_statuses = {}
        overall_health = 0
        n_agents = 0
        
        for agent_name, history in self.history.items():
            if history:
                latest = history[-1]
                health = latest.health_score()
                agent_statuses[agent_name] = {
                    'health': health,
                    'f1': latest.f1,
                    'n_evaluations': len(history),
                }
                overall_health += health
                n_agents += 1
        
        return {
            'overall_health': overall_health / max(n_agents, 1),
            'n_agents': n_agents,
            'n_alerts': len(self.alerts),
            'recent_alerts': self.alerts[-5:],
            'agents': agent_statuses,
        }


# Create and test self-evaluation agent
self_eval = SelfEvaluationAgent()

for agent_name, (y_true, y_pred, y_prob) in agents_data.items():
    # Evaluate in batches to simulate streaming
    batch_size = 200
    for start in range(0, len(y_true), batch_size):
        end = min(start + batch_size, len(y_true))
        report = self_eval.evaluate_and_record(
            agent_name,
            y_true[start:end],
            y_pred[start:end],
            y_prob[start:end],
        )

# Get system status
status = self_eval.get_system_status()
print(f'\n{"="*60}')
print(f'SYSTEM HEALTH: {status["overall_health"]:.1f}/100')
print(f'{"="*60}')
for agent, info in status['agents'].items():
    print(f'  {agent:20s} Health={info["health"]:.1f} F1={info["f1"]:.4f}')
print(f'\nAlerts: {status["n_alerts"]}')
for alert in status['recent_alerts']:
    print(f'  [{alert["level"]:8s}] {alert["agent"]}: {alert["message"]}')

In [ ]:
# Visualization: Agent health over time
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Health scores bar chart
ax = axes[0]
status = self_eval.get_system_status()
agents_list = list(status['agents'].keys())
healths = [status['agents'][a]['health'] for a in agents_list]
colors = ['#4CAF50' if h >= 75 else '#FF9800' if h >= 60 else '#F44336' for h in healths]
bars = ax.barh(agents_list, healths, color=colors, alpha=0.8)
ax.axvline(75, color='orange', linestyle='--', alpha=0.5, label='Warning')
ax.axvline(60, color='red', linestyle='--', alpha=0.5, label='Critical')
ax.set_xlabel('Health Score'); ax.set_title('Agent Health Scores')
ax.legend(); ax.set_xlim(0, 105); ax.grid(True, alpha=0.3, axis='x')

# Health history over evaluations
ax = axes[1]
for agent_name, history in self_eval.history.items():
    if history:
        healths = [m.health_score() for m in history]
        ax.plot(healths, label=agent_name, linewidth=2)
ax.axhline(75, color='orange', linestyle='--', alpha=0.5)
ax.axhline(60, color='red', linestyle='--', alpha=0.5)
ax.set_xlabel('Evaluation #'); ax.set_ylabel('Health Score')
ax.set_title('Health Over Time'); ax.legend(); ax.grid(True, alpha=0.3)

plt.suptitle('Self-Evaluation Dashboard', fontsize=14, fontweight='bold')
plt.tight_layout(); plt.show()

---
## 8. Feature Drift Detection Demo

In [ ]:
# Simulate feature drift
n_features = 20
n_ref = 500
n_new = 200

# Reference data (training distribution)
ref_data = np.random.randn(n_ref, n_features)

# New data with drift in some features
new_data = np.random.randn(n_new, n_features)
# Inject drift in features 3, 7, 12
new_data[:, 3] += 1.5  # Mean shift
new_data[:, 7] *= 2.0  # Variance change
new_data[:, 12] += 0.8  # Slight shift

# Detect drift
drift_detector = FeatureDriftDetector(ref_data)
drift_result = drift_detector.check_drift(new_data)

print(f'Features drifted: {drift_result["n_features_drifted"]}/{drift_result["total_features"]}')
print(f'Drift ratio: {drift_result["drift_ratio"]:.3f}')
print(f'Overall drift detected: {drift_result["overall_drift"]}')

# Show drifted features
print('\nDrifted features:')
for detail in drift_result['details']:
    if detail['drift']:
        print(f'  Feature {detail["feature_idx"]}: KS={detail["ks_stat"]:.4f}, p={detail["p_value"]:.6f}')

In [ ]:
# Visualize drifted features
drifted_features = [d['feature_idx'] for d in drift_result['details'] if d['drift']]
non_drifted = [d['feature_idx'] for d in drift_result['details'] if not d['drift']][:3]
features_to_plot = drifted_features[:4] + non_drifted[:2]

fig, axes = plt.subplots(2, 3, figsize=(16, 8))
axes = axes.ravel()

for ax, feat_idx in zip(axes, features_to_plot):
    is_drifted = feat_idx in drifted_features
    detail = drift_result['details'][feat_idx]
    
    ax.hist(ref_data[:, feat_idx], bins=30, alpha=0.5, label='Reference', color='#2196F3', density=True)
    ax.hist(new_data[:, feat_idx], bins=30, alpha=0.5, label='New', color='#F44336' if is_drifted else '#4CAF50', density=True)
    
    status = 'DRIFT' if is_drifted else 'OK'
    ax.set_title(f'Feature {feat_idx} [{status}]\nKS={detail["ks_stat"]:.3f}, p={detail["p_value"]:.4f}',
                fontsize=10)
    ax.legend(fontsize=8); ax.grid(True, alpha=0.3)

plt.suptitle('Feature Distribution Drift Detection', fontsize=14, fontweight='bold')
plt.tight_layout(); plt.show()

---
## 9. Save & Export

In [ ]:
# Save evaluation results
evaluation_report = {
    'system_status': self_eval.get_system_status(),
    'agent_metrics': {name: metrics.to_dict() for name, metrics in all_metrics.items()},
    'deception_metrics': {
        'deception_success_rate': dec_metrics.deception_success_rate,
        'real_asset_protection': dec_metrics.real_asset_protection_rate,
        'detection_latency': dec_metrics.detection_latency,
        'overall_effectiveness': dec_metrics.overall_effectiveness(),
    },
    'alerts': self_eval.alerts,
}

with open(EVAL_DIR / 'evaluation_report.json', 'w') as f:
    json.dump(evaluation_report, f, indent=2, default=str)

print(f'Evaluation report saved to {EVAL_DIR}')
print(f'\n✓ Notebook 08 complete. Ready for Notebook 09 (Agent Orchestration).')